# Data exploration for AstroMAE

This notebook walks through inspecting the preprocessed dataset, running a trained checkpoint on a few samples, and plotting a reconstruction grid.

Before running the cells below, preprocess (or synthesise) some data:

```bash
astromae data preprocess --synthetic 16 --synthetic-classes 3 --image-size 32
astromae train --config configs/tiny.yaml --epochs 1
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from astromae.data.dataset import PreprocessedAstroDataset
from astromae.evaluation.embed import extract_embeddings, load_model_from_checkpoint
from astromae.evaluation.probe import cluster_embeddings, linear_probe
from astromae.evaluation.visualize import save_reconstruction_grid

## 1. Inspect the dataset

In [ ]:
PROCESSED_DIR = Path("../data/processed")
dataset = PreprocessedAstroDataset(PROCESSED_DIR)
print("num samples =", len(dataset))
print("classes     =", dataset.classes)

sample, label = dataset[0]
print("sample shape =", tuple(sample.shape), "label =", label)

plt.imshow(sample.permute(1, 2, 0).clamp(0, 1).numpy())
plt.title(dataset.classes[label])
plt.axis("off")
plt.show()

## 2. Load a checkpoint and extract embeddings

In [ ]:
CKPT = Path("../outputs/checkpoints/last.pt")
model, cfg = load_model_from_checkpoint(CKPT)
emb, labels, classes = extract_embeddings(model, PROCESSED_DIR, batch_size=4)
print("embeddings:", emb.shape)

## 3. Downstream evaluation

In [ ]:
probe = linear_probe(emb, labels, test_size=0.3, seed=0)
print(f"linear probe accuracy: {probe.accuracy:.3f}, macro-F1: {probe.macro_f1:.3f}")

clusters = cluster_embeddings(emb, labels, seed=0)
print("clustering:", clusters)

## 4. Reconstruction grid

In [ ]:
samples = torch.stack([dataset[i][0] for i in range(min(6, len(dataset)))], dim=0)
path = save_reconstruction_grid(
    model, samples, "../outputs/reconstruction.png", mask_ratio=cfg.mask_ratio
)
print("saved to", path)